# UCI Sticky-Sampler Sparsity Ablation — PIW Sweep

Sticky Boomerang vs. Sticky ZigZag on Boston, swept over the spike-and-slab prior inclusion
weight $w$, for two hidden-layer variants (`shallow`, `deep_narrow`) loaded side by side.
Produces a metrics table, an event-type breakdown table, and figures covering
sparsity/surviving-weight behaviour, posterior noise, and per-weight inclusion probability.


In [ ]:
def make_data(N: int, D: int, n_signals: int, signal_scale: float,
              noise_std: float, seed: int):
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(N, D, generator=g, dtype=torch.float64)
    beta_true = torch.zeros(D, dtype=torch.float64)
    # Geometrically decaying magnitudes with alternating signs: the largest
    # signals are unambiguous (P(gamma=1|y) ~ 1) while the smallest sit near
    # the noise floor, producing INTERMEDIATE inclusion probabilities. Those
    # are the coordinates that actually discriminate between samplers -- a
    # scatter with mass only at 0 and 1 would validate very little.
    signs = torch.tensor([1.0 if i % 2 == 0 else -1.0 for i in range(n_signals)], dtype=torch.float64)
    decay = torch.tensor([0.5 ** i for i in range(n_signals)], dtype=torch.float64)
    beta_true[:n_signals] = signal_scale * signs * decay
    y = X @ beta_true + noise_std * torch.randn(N, generator=g, dtype=torch.float64)
    return X, y, beta_true

X, y, beta = make_data(N=200, D=20, n_signals=5, signal_scale=1.5, noise_std=1.0, seed=0)
print(beta)

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. Load runs and predictions

Discovers every `piw_*/` directory under `results/paper/uci_sparsity_ablation/<variant>/<dataset>/split_XX/`
for each hidden-layer variant, then computes the per-draw predictive mean
$f_s(x) = \mathrm{NN}_{\beta_s}(x)$ and noise $\sigma_s = \exp(\log\sigma_s)$ for each of the
$S$ resampled posterior draws $\beta_s = (\text{weights}_s, \log\sigma_s)$.

Both variants load into a single `runs` dict keyed by `(variant, w, stem)`, so every downstream
cell can slice by variant, PIW, or sampler without juggling separate globals.


In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)
from sazz.gpu_friendly.scripts.uci_sparsity_ablation import build_bm_only

DATASET, SPLIT_ID = "boston", 0
OUT_MAPS_DIR = Path("results/maps/uci_sparsity_ablation")
OUT_DIR = Path("results/paper/uci_sparsity_ablation")
MAP_CKPT_PATH = OUT_MAPS_DIR / f"{DATASET}_split{SPLIT_ID:02d}_map.pt"

# variant -> root dir holding piw_*/ subdirs for (DATASET, SPLIT_ID)
VARIANT_ROOTS = {
    "shallow": OUT_DIR / "shallow" / DATASET / f"split_{SPLIT_ID:02d}",
    "deep_narrow": OUT_DIR / "deep_narrow" / DATASET / f"split_{SPLIT_ID:02d}",
}
VARIANT_LS = {"shallow": "-", "deep_narrow": "--"}  # linestyle used in every figure below

LABELS = {"grid_sticky_zigzag": "Sticky ZigZag", "grid_sticky_boomerang": "Sticky Boomerang"}
COLORS = {"grid_sticky_zigzag": "C2", "grid_sticky_boomerang": "C0"}

raw = load_raw_datasets((DATASET,))
data = make_split(*raw[DATASET], seed=BASE_SEED + SPLIT_ID, dtype=DTYPE, device=DEVICE)
X_test, y_test, y_std = data["X_test"], data["y_test"], data["y_std"]

map_ckpt = torch.load(MAP_CKPT_PATH, map_location="cpu", weights_only=False)


def discover_piw_files(root: Path, require_suffix: str | None) -> dict[float, dict[str, Path]]:
    """{w: {stem: path}} for every piw_<w>/ dir under root holding matching .pt files."""
    piw_dirs: dict[float, dict[str, Path]] = {}
    for p in sorted(root.glob("piw_*")):
        pattern = f"*{require_suffix}.pt" if require_suffix else "*.pt"
        files = [f for f in p.glob(pattern) if f.stem.removesuffix(require_suffix or "") in LABELS]
        if not files:
            continue
        w = float(p.name.removeprefix("piw_"))
        piw_dirs[w] = {f.stem.removesuffix(require_suffix or ""): f for f in files}
    return piw_dirs


@torch.no_grad()
def predict_all(bm_, weight_samples: Tensor, X_new: Tensor) -> Tensor:
    """f_s(X_new) for every posterior draw s -- [S, N]."""
    return torch.stack([
        torch.func.functional_call(bm_.module, bm_.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])


# One `bm` (module + param layout) per variant, plus per-variant metadata for later cells.
VARIANT_BM: dict[str, Any] = {}
VARIANT_META: dict[str, dict] = {}

shallow_cfg = BNNConfig(layer_sizes=map_ckpt["layer_sizes"], activation=map_ckpt["activation"],
                         prior_sigma_scale=map_ckpt["prior_sigma_scale"])
VARIANT_BM["shallow"] = build_bm_only(data, shallow_cfg)
VARIANT_META["shallow"] = dict(layer_sizes=map_ckpt["layer_sizes"], activation=map_ckpt["activation"],
                                has_frozen_mask=True)

# runs: {(variant, w, stem): {payload, diag_df, preds, mean_pred, noise_samples, weight_samples, ...}}
runs: dict[tuple[str, float, str], dict] = {}

for variant, root in VARIANT_ROOTS.items():
    require_suffix = "_skeleton" if variant == "shallow" else None
    piw_dirs = discover_piw_files(root, require_suffix)
    assert piw_dirs, f"No .pt files found under {root}. Run uci_sparsity_ablation.py first."

    for w, stems in sorted(piw_dirs.items()):
        for stem, f in stems.items():
            payload = torch.load(f, map_location="cpu", weights_only=False)
            diag_log = payload.get("diagnostics")
            runs[(variant, w, stem)] = dict(
                payload=payload, diag_df=pd.DataFrame(diag_log) if diag_log else None,
            )

    if variant != "shallow":
        # deep_narrow (and any future variant): no shared MAP checkpoint, build bm from the
        # payload's own layer_sizes/prior_sigma_scale (assumed identical across its runs).
        any_payload = next(r["payload"] for (v, _, _), r in runs.items() if v == variant)
        cfg = BNNConfig(layer_sizes=any_payload["layer_sizes"], activation=any_payload["activation"],
                         prior_sigma_scale=any_payload["prior_sigma_scale"])
        VARIANT_BM[variant] = build_bm_only(data, cfg)
        VARIANT_META[variant] = dict(layer_sizes=any_payload["layer_sizes"],
                                      activation=any_payload["activation"], has_frozen_mask=False)

    n_w = len({w for (v, w, _), _ in runs.items() if v == variant})
    n_runs = sum(1 for v, _, _ in runs if v == variant)
    print(f"{DATASET} split_{SPLIT_ID:02d} {variant} | D={VARIANT_BM[variant].D} | "
          f"{n_runs} runs across {n_w} PIW values")

for (variant, w, stem), r in runs.items():
    bm_ = VARIANT_BM[variant]
    samples = r["payload"]["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
    r["weight_samples"] = samples[:, :-1]
    r["preds"] = predict_all(bm_, r["weight_samples"], X_test)     # [S, N]
    r["mean_pred"] = r["preds"].mean(0)
    r["epist_std"] = r["preds"].std(0)
    r["noise_samples"] = samples[:, -1].exp()                      # [S]
    r["noise_std_eff"] = float(r["noise_samples"].mean())
    r["total_std"] = (r["epist_std"] ** 2 + r["noise_std_eff"] ** 2).sqrt()
    r["achieved_sparsity"] = float((r["weight_samples"] == 0).float().mean())

## 2. Metrics table

Posterior predictive is the $S$-draw Gaussian mixture $p(y\mid x) = \frac1S\sum_s
\mathcal N(y; f_s(x), \sigma_s^2)$. Reported per $(\text{variant}, w, \text{sampler})$:

$$\mathrm{RMSE} = \sqrt{\textstyle\frac1N\sum_n (\bar f(x_n) - y_n)^2}\,\hat\sigma_y,
\qquad \bar f = \frac1S\sum_s f_s$$
$$\mathrm{NLL} = -\frac1N\sum_n \log\Big(\frac1S\sum_s \mathcal N(y_n; f_s(x_n), \sigma_s^2)\Big)
+ \log\hat\sigma_y$$

CRPS is the closed-form Gaussian-mixture CRPS (Gneiting & Raftery 2007), evaluated exactly
for pairs of mixture components. Coverage uses the two-moment Gaussian approximation to the
mixture, $\mathcal N(\bar f, \,\mathrm{Var}_s[f_s] + \overline{\sigma^2})$.


In [ ]:
CRPS_MIX_SUBSAMPLE = 256  # CRPS is O(S^2) -- subsample draws for speed

def _crps_gauss_term(m: Tensor, s: Tensor) -> Tensor:
    """E|X - m| for X ~ N(0, s^2)."""
    d = Normal(0.0, 1.0)
    z = m / s
    return m * (2 * d.cdf(z) - 1) + 2 * s * d.log_prob(z).exp()

def compute_rmse(mean_pred): return float(((mean_pred - y_test) ** 2).mean().sqrt()) * y_std

def compute_nll_mixture(preds, noise):
    lp = Normal(preds, noise[:, None]).log_prob(y_test)
    return float(-(torch.logsumexp(lp, 0) - math.log(preds.shape[0])).mean() + math.log(y_std))

def compute_crps_mixture(preds, noise, n_sub=CRPS_MIX_SUBSAMPLE, seed=0):
    idx = torch.randperm(preds.shape[0], generator=torch.Generator().manual_seed(seed))[:n_sub]
    mu, sd = preds[idx] * y_std, (noise[idx] * y_std)[:, None].expand(-1, preds.shape[1])
    yt = y_test * y_std
    term1 = _crps_gauss_term(yt[None] - mu, sd).mean(0)
    diff = mu[:, None] - mu[None, :]
    s2 = (sd[:, None] ** 2 + sd[None, :] ** 2).sqrt()
    term2 = _crps_gauss_term(diff, s2).mean((0, 1))
    return float((term1 - 0.5 * term2).mean())

def compute_coverage(mean_pred, total_std, level):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_test - mean_pred).abs() <= z * total_std).float().mean())

rows = []
for (variant, w, stem), r in runs.items():
    rows.append({
        "Variant": variant, "PIW": w, "Sampler": LABELS.get(stem, stem),
        "RMSE": compute_rmse(r["mean_pred"]),
        "NLL": compute_nll_mixture(r["preds"], r["noise_samples"]),
        "CRPS": compute_crps_mixture(r["preds"], r["noise_samples"]),
        "Cov 90%": compute_coverage(r["mean_pred"], r["total_std"], 0.90),
        "Cov 95%": compute_coverage(r["mean_pred"], r["total_std"], 0.95),
        "Achieved sparsity": r["achieved_sparsity"],
    })
metrics_df = pd.DataFrame(rows).sort_values(["Variant", "PIW", "Sampler"]).set_index(["Variant", "PIW", "Sampler"])

def _highlight_coverage(col):
    target = 0.90 if "90" in col.name else 0.95
    best = (col - target).abs().idxmin()
    return ["background-color: #68dc0f" if idx == best else "" for idx in col.index]

metrics_df.style \
    .highlight_min(subset=["RMSE", "NLL", "CRPS"], color="#68dc0f") \
    .apply(_highlight_coverage, subset=["Cov 90%", "Cov 95%"]) \
    .format(precision=4)

## 3. Event-type breakdown

Every sampler-loop iteration ends in exactly one event: **bounce** (Poisson-thinning
candidate accepted, velocity reflects), **no_event** (thinning rejected the whole grid
window), **freeze**/**thaw** (a coordinate hits/leaves zero, sticky-only), or **refresh**
(velocity fully resampled, Boomerang-only). Table entries are each event type's share of
total loop iterations, $(\text{variant}, w, \text{sampler})$-wise.


In [ ]:
EVENT_TYPES = ["bounce", "no_event", "freeze", "thaw", "refresh"]

event_rows = []
for (variant, w, stem), r in runs.items():
    if r["diag_df"] is None:
        continue
    if w in [0.01, 0.05, 0.2, 0.4, 0.6, 0.8]:
        continue
    row = {"Variant": variant, "PIW": w, "Sampler": LABELS.get(stem, stem)}
    row.update({e: (r["diag_df"]["event_type"] == e).mean() for e in EVENT_TYPES})
    event_rows.append(row)

event_df = pd.DataFrame(event_rows).sort_values(["PIW", "Sampler"]).set_index(["PIW", "Sampler"])
event_df.style.format(precision=3).background_gradient(cmap="Blues", axis=None)

## 4. Figure 1 — achieved sparsity and surviving-weight magnitude vs. nominal $w$

**(a)** achieved sparsity $\frac1D\sum_i \mathbb 1[\hat\beta_i = 0]$ against the nominal
prior weight $w$. **(b)** mean magnitude of the surviving (non-zero posterior-mean) weights,
also against nominal $w$. Color denotes sampler, linestyle denotes hidden-layer variant
(solid = `shallow`; dashed = `deep_narrow`).

In [ ]:
LABELS = {"grid_sticky_zigzag": "Sticky ZigZag", "grid_sticky_boomerang": "Sticky Boomerang"}
COLORS = {"grid_sticky_zigzag": "C2", "grid_sticky_boomerang": "C0"}

fig, (ax_sp, ax_horizon) = plt.subplots(1, 2, figsize=(12, 5))
stems = list(LABELS.keys())
i=0
for variant, ls in VARIANT_LS.items():
    for stem in stems:
        i+=1
        color = COLORS[stem]
        keys = sorted(k for k in runs if k[0] == variant and k[2] == stem)
        if not keys:
            continue
        #label = f"{LABELS[stem]}"
        label = LABELS[stem] if variant == "shallow" else "_nolegend_"
        ws = [w for _, w, _ in keys]
        sp = [runs[k]["achieved_sparsity"] for k in keys]
        sp_percentage = []
        for i in range (len(sp)):
            sp_percentage.append(sp[i]*100)
            #print(sp[i]*100)
            
        ax_sp.plot(ws, sp_percentage, marker="o", ms=5, color=color, ls=ls, label=label)


for stem in stems:
    color = COLORS[stem]
    keys = sorted(k for k in runs if k[0] == "shallow" and k[2] == stem and runs[k]["diag_df"] is not None)
    if not keys:
        continue
    ws = [w for _, w, _ in keys]
    mean_horizon = [float(runs[k]["diag_df"]["horizon"].mean()) for k in keys]
    ax_horizon.plot(ws, mean_horizon, marker="o", ms=5, color=color, ls="-", label=f"{LABELS[stem]}")

ax_sp.set(xlabel="$w$", ylabel="Achieved sparsity %", 
          title="Achieved sparsity",
          xlim=(0, 1), ylim=(0, 100))
ax_horizon.set(xlabel="$w$", ylabel="Mean realized horizon",
               title="Realized step horizon", 
               xlim=(0, 1), yscale="log")
for ax in (ax_sp, ax_horizon):
    ax.legend(fontsize=7)

#fig.suptitle(f"{DATASET.capitalize()}, sparsity and sampler step horizon across the PIW sweep", y=1.02)
plt.tight_layout()
plt.savefig(f"results/plots/sparsity_ablation/{DATASET}.pdf", bbox_inches="tight", dpi=200)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(stems), figsize=(6 * len(stems), 4.3), sharey=False)
W_COMPARE = [0.1, 0.9]
W_STYLES = {0.1: "-", 0.9: "--"}

for ax, stem in zip(np.atleast_1d(axes), stems):
    for variant in VARIANT_LS:
        for w in W_COMPARE:
            key = (variant, w, stem)
            if key not in runs:
                continue
            tmax_log = runs[key]["payload"]["grid_t_max_log"]
            alpha = 1.0 if variant == "shallow" else 0.6
            ax.plot(tmax_log, color=COLORS[stem], ls=W_STYLES[w], alpha=alpha,
                    label=f"{variant}, $w$={w:g}")
    ax.set(xlabel="Sampler iteration", ylabel=r"Adapted grid horizon $t_\max$",
           title=LABELS[stem], yscale="log")
    ax.legend(fontsize=7)

fig.suptitle(f"{DATASET.capitalize()}, $t_\\max$ adaptation trajectory at $w=0.1$ vs. $w=0.9$", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.3))

for variant, ls in VARIANT_LS.items():
    for stem in stems:
        color = COLORS[stem]
        keys = sorted(k for k in runs if k[0] == variant and k[2] == stem)
        if not keys:
            continue
        label = f"{LABELS[stem]} ({variant})"

        pts = []
        for k in keys:
            r = runs[k]
            w_mean = r["weight_samples"].mean(0)
            w_std = r["weight_samples"].std(0)
            alive = w_mean != 0
            unc = float(w_std[alive].mean()) if alive.any() else float("nan")
            pts.append((r["achieved_sparsity"], unc))
        pts.sort()
        sp_x, unc = map(np.array, zip(*pts))
        ax.plot(sp_x, unc, marker="o", ms=5, color=color, ls=ls, label=label)

ax.set(xlabel="Achieved sparsity", ylabel="Mean posterior std (surviving weights)",
       title=f"{DATASET.capitalize()}, posterior uncertainty on surviving weights vs. sparsity",
       xlim=(0, 1))
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"sparsity_uncertainty_{DATASET}.pdf", bbox_inches="tight", dpi=200)
plt.show()

## 5. Sparse network

In [ ]:
# --- Posterior inclusion probability for one (variant, PIW, sampler), sliced onto the network ---
import matplotlib as mpl

VARIANT_VIS = "shallow"           # "shallow" or "deep_narrow"
STEM        = "grid_sticky_zigzag"   # "grid_sticky_zigzag" or "grid_sticky_boomerang"
PIW_VIS     = 0.01                     # which point of the sweep to visualise
ZERO_TOL    = 0.0                     # exact-zero test; use e.g. 1e-8 for near-zero

vis_key = (VARIANT_VIS, PIW_VIS, STEM)
assert vis_key in runs, f"no run for {vis_key}; have {sorted(runs)}"
r = runs[vis_key]

# weight_samples is [S, D-1] -- network weights + biases only, log_sigma already dropped
ws_arr    = r["weight_samples"].numpy()
S, Dw     = ws_arr.shape
incl_prob = (np.abs(ws_arr) > ZERO_TOL).mean(axis=0)          # P(active), one per network coord

layer_sizes = VARIANT_META[VARIANT_VIS]["layer_sizes"]          # e.g. [13, 50, 1]
activation  = VARIANT_META[VARIANT_VIS]["activation"]
n_net = sum(n_in * n_out + n_out for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]))
assert n_net == Dw, (n_net, Dw)                                # weights+biases, no log_sigma

# Walk named_parameters() order: layers.i.weight [n_out, n_in], then layers.i.bias [n_out]
W_incl, b_incl, off = [], [], 0
for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
    W_incl.append(incl_prob[off:off + n_out * n_in].reshape(n_out, n_in)); off += n_out * n_in
    b_incl.append(incl_prob[off:off + n_out]);                            off += n_out
assert off == Dw, (off, Dw)

print(f"{VARIANT_VIS} {LABELS[STEM]}  w={PIW_VIS:g}   (S={S} draws)")
for li, (Wp, bp) in enumerate(zip(W_incl, b_incl)):
    print(f"  layer {li}: W{Wp.shape}  mean incl={Wp.mean():.3f}   "
          f"bias mean incl={bp.mean():.3f}   fully-excluded weights={(Wp == 0).mean():.3f}")

In [ ]:
# --- Draw the network, edges coloured by posterior inclusion probability ---
cmap = mpl.colormaps["RdYlGn"]                  # 0 -> red (excluded), 1 -> green (included)
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

xs = np.arange(len(layer_sizes))
def _ys(n):
    return np.linspace(0, 1, n) if n > 1 else np.array([0.5])
node_y = [_ys(n) for n in layer_sizes]

fig, ax = plt.subplots(figsize=(4 + 1.6 * len(layer_sizes), 9))

# edges: weight[j, k] connects node k in layer li to node j in layer li+1
for li, Wp in enumerate(W_incl):
    n_out, n_in = Wp.shape
    y0, y1 = node_y[li], node_y[li + 1]
    order = np.argsort(Wp.ravel())             # draw excluded (red) first, included (green) on top
    for idx in order:
        j, k = divmod(idx, n_in)
        p = Wp[j, k]
        ax.plot([xs[li], xs[li + 1]], [y0[k], y1[j]],
                color=cmap(norm(p)), lw=0.4 + 2.2 * p,
                alpha=0.15 + 0.85 * p, zorder=1, solid_capstyle="round")

# nodes coloured by mean incoming-weight inclusion (input layer: neutral grey)
for li, n in enumerate(layer_sizes):
    node_c = ["0.6"] * n if li == 0 else [cmap(norm(W_incl[li - 1][j].mean())) for j in range(n)]
    ax.scatter(np.full(n, xs[li]), node_y[li], s=260, c=node_c,
               edgecolors="black", linewidths=0.8, zorder=3)

labels = ([f"input\n({layer_sizes[0]})"]
          + [f"hidden {i}\n({s}, {activation})" for i, s in enumerate(layer_sizes[1:-1], 1)]
          + [f"output\n({layer_sizes[-1]})"])
for x, lab in zip(xs, labels):
    ax.text(x, -0.08, lab, ha="center", va="top", fontsize=11)

ax.set_xlim(xs[0] - 0.3, xs[-1] + 0.3); ax.set_ylim(-0.15, 1.05); ax.axis("off")
ax.set_title(f"{VARIANT_VIS} {LABELS[STEM]}, {DATASET.capitalize()}  (w = {PIW_VIS:g}) — "
             f"posterior inclusion probability per weight\n"
             f"(green = almost always active, red = almost always pruned; "
             f"line weight = certainty)", fontsize=12)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
             fraction=0.03, pad=0.02, label="P(weight active)")
plt.tight_layout()
plt.show()
# fig.savefig(f"network_inclusion_{DATASET}_{VARIANT_VIS}_{STEM}_w{PIW_VIS:g}.pdf", bbox_inches="tight")

In [ ]:
# --- Keep only weights active > cutoff, re-evaluate on the test set (same (variant, PIW, sampler)) ---
INCL_CUTOFF = 0.90

r          = runs[vis_key]
bm_vis     = VARIANT_BM[VARIANT_VIS]
samples    = r["payload"]["samples"].to(dtype=DTYPE)          # [S, D]  (last col = log_sigma)
weight_s   = r["weight_samples"]                              # [S, D-1]
keep_mask  = torch.as_tensor(incl_prob > INCL_CUTOFF, dtype=weight_s.dtype)   # [D-1]

weight_masked = weight_s * keep_mask                         # zero the <=cutoff coords in every draw

preds_m   = predict_all(bm_vis, weight_masked, X_test)        # [S, N]
mean_m    = preds_m.mean(0)
noise_m   = samples[:, -1].exp()                             # [S]  (log_sigma untouched by the mask)
total_m   = (preds_m.std(0) ** 2 + float(noise_m.mean()) ** 2).sqrt()

n_kept = int(keep_mask.sum())
print(f"{VARIANT_VIS} {LABELS[STEM]}  w={PIW_VIS:g}: kept {n_kept} / {weight_s.shape[1]} weights "
      f"active >{INCL_CUTOFF:.0%} of the time ({n_kept / weight_s.shape[1]:.1%})\n")
print(f"  RMSE    : {compute_rmse(mean_m):.4f}   "
      f"(full posterior: {compute_rmse(r['mean_pred']):.4f})")
print(f"  NLL     : {compute_nll_mixture(preds_m, noise_m):.4f}   "
      f"(full posterior: {compute_nll_mixture(r['preds'], r['noise_samples']):.4f})")
print(f"  CRPS    : {compute_crps_mixture(preds_m, noise_m):.4f}   "
      f"(full posterior: {compute_crps_mixture(r['preds'], r['noise_samples']):.4f})")
print(f"  Cov 90% : {compute_coverage(mean_m, total_m, 0.90):.3f}   "
      f"(full: {compute_coverage(r['mean_pred'], r['total_std'], 0.90):.3f})")
print(f"  Cov 95% : {compute_coverage(mean_m, total_m, 0.95):.3f}   "
      f"(full: {compute_coverage(r['mean_pred'], r['total_std'], 0.95):.3f})")